# Experiment vs observation

**STA 509 2.0 Introduction to Design and Analysis of Experiments** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/sta509.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Experimental vocabulary](#theory-01)
- [Randomization, replication and local control](#theory-02)
- [CRD allocation](#theory-03)
- [Model, constraints and hypotheses](#theory-04)
- [Sums of squares and computational formulas](#theory-05)
- [Meaning of the result](#theory-06)
- [Strengths and limitations](#theory-07)
- [Extension in the outline: two-factor studies](#theory-08)

## Detailed theory · Completely randomized design and model building

<a id="theory-01"></a>

### Experimental vocabulary
An experimental unit independently receives a treatment. A response is the measured outcome; a factor is an explanatory condition; a level is one setting; a treatment is a level or level combination. Repeated readings on one unit improve measurement precision but do not automatically supply independent replication.

<a id="theory-02"></a>

### Randomization, replication and local control
Randomization prevents systematic assignment according to unmeasured influences in expectation and supports the inference mechanism. Replication estimates experimental variation. Local control reduces known nuisance variation by blocking. Randomization does not guarantee perfect finite-sample balance, and replication does not remove bias from a confounded assignment.

<a id="theory-03"></a>

### CRD allocation
In a CRD, allocate treatments at random over all eligible units, respecting planned replication. It is most useful with reasonably homogeneous units and no important known blocking structure. Record the randomization procedure and run order. Random treatment assignment and random population sampling support different generalizations.

<a id="theory-04"></a>

### Model, constraints and hypotheses
For a fixed treatment factor,
$$Y_{ij}=\mu+\tau_i+\epsilon_{ij},\quad E(\epsilon_{ij})=0,\quad Var(\epsilon_{ij})=\sigma^2.$$
Use an identifiability constraint such as $\sum\tau_i=0$ under a balanced sum-to-zero parameterization, or a reference-category model. The hypothesis is equality of all treatment means, not equality of every observed response. For exact classical F inference, errors are independent, normal and common variance.

<a id="theory-05"></a>

### Sums of squares and computational formulas
Let $T_i=\sum_jy_{ij}$, $G=\sum_iT_i$ and $N=\sum_i n_i$. The correction factor is $CF=G^2/N$:
$$SS_{Total}=\sum_{ij}y_{ij}^2-CF,\quad
SS_{Trt}=\sum_i T_i^2/n_i-CF,\quad SS_E=SS_{Total}-SS_{Trt}.$$
These equal the centred deviation formulas. Treatment, error and total df are a−1,N−a,N−1. Mean squares divide SS by their own df, and $F=MS_{Trt}/MS_E$ uses an upper F tail.

<a id="theory-06"></a>

### Meaning of the result
Large F provides evidence that at least one population mean differs. It does not rank every treatment or establish all pairwise differences. Follow-up comparisons and model checks are part of interpretation, not consequences automatically supplied by significance.

<a id="theory-07"></a>

### Strengths and limitations
CRD is flexible, simple and can handle unequal replication under the usual model. Ignoring a strong nuisance gradient can inflate error; poor replication limits diagnostics and power. Choosing a design must occur before treatment assignment, rather than retroactively calling unrelated observations an experiment.

<a id="theory-08"></a>

### Extension in the outline: two-factor studies
For a balanced crossed design with a levels of A, b of B and r independent replicates per cell,
$$Y_{ijk}=\mu+\alpha_i+\beta_j+(\alpha\beta)_{ij}+\epsilon_{ijk}.$$
Main effects average over the other factor. Interaction means one factor's effect depends on the other's level. The df are a−1,b−1,(a−1)(b−1),ab(r−1) for A,B,interaction,error.
$$SS_A=br\sum_i(\bar y_{i..}-\bar y_{...})^2,\quad
SS_B=ar\sum_j(\bar y_{.j.}-\bar y_{...})^2,$$
$$SS_{AB}=r\sum_{ij}(\bar y_{ij.}-\bar y_{i..}-\bar y_{.j.}+\bar y_{...})^2.$$
Within-cell residuals supply SSE. In the fixed-effects common-variance model, divide each factor/interaction MS by MSE. Interpret important interaction before marginal main effects. With one observation per cell, a full interaction model has no independent pure-error df; an additive analysis requires its no-interaction assumption.

## Concepts covered

**Course points:** Experiment vs observation; response/factors/treatments/units; randomization, replication; completely randomized design; one-way ANOVA.

**Mission:** Predict what the formula will do, then run the simulation.

## Worked calculation · Completely randomized design: full ANOVA working table

### Question and assumptions

Three treatments have independent responses A=[8,9,10], B=[10,12,14], C=[13,14,15]. Use the fixed-effects one-way ANOVA model and $\alpha=.05$.

### Procedure

1. Show treatment totals, means and within-group sums of squares.
2. Calculate the grand mean and between-treatment SS.
3. Fill SS, df, MS and F columns, then compare the treatment F to the correct upper-tail reference.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Treatment | Responses | n | Total | Mean | Within SS | Between SS contribution |
| --- | --- | --- | --- | --- | --- | --- |
| A | [8, 9, 10] | 3 | 27 | 9 | 2 | 21.3333 |
| B | [10, 12, 14] | 3 | 36 | 12 | 8 | 0.333333 |
| C | [13, 14, 15] | 3 | 42 | 14 | 2 | 16.3333 |

| Source | SS | df | MS | F |
| --- | --- | --- | --- | --- |
| Treatment | 38 | 2 | 19 | 9.5 |
| Error | 12 | 6 | 2 |  |
| Total | 50 | 8 |  |  |

| Decision quantity | Value |
| --- | --- |
| Grand mean | 11.6667 |
| F(.95;2,6) cutoff | 5.14325 |
| Decision | Reject equal means |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


groups={"A":[8,9,10],"B":[10,12,14],"C":[13,14,15]}
all_values=[v for values in groups.values() for v in values]; grand=mean(all_values); N=len(all_values); a=len(groups)
rows=[]
for name,values in groups.items():
    xb=mean(values); within=sum((v-xb)**2 for v in values); between=len(values)*(xb-grand)**2
    rows.append([name,values,len(values),sum(values),xb,within,between])
show_table(["Treatment","Responses","n","Total","Mean","Within SS","Between SS contribution"],rows)
SST=sum((v-grand)**2 for v in all_values); SStr=sum(r[6] for r in rows); SSE=sum(r[5] for r in rows)
MST=SStr/(a-1); MSE=SSE/(N-a); F=MST/MSE
show_table(["Source","SS","df","MS","F"],[["Treatment",SStr,a-1,MST,F],["Error",SSE,N-a,MSE,""],["Total",SST,N-1,"",""]])
show_table(["Decision quantity","Value"],[["Grand mean",grand],["F(.95;2,6) cutoff",5.143252849784718],["Decision","Reject equal means" if F>5.143252849784718 else "Do not reject"]])
assert abs(SST-SStr-SSE)<1e-12 and abs(F-9.5)<1e-12


### Decision and interpretation

The significant omnibus F says at least one population mean differs; it does not identify all differing pairs. Independence comes from the experimental design. Check residual distributions and group spreads before relying on small-sample normal-theory inference.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
rng=np.random.default_rng(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
groups=[rng.normal(mu,4,18) for mu in [18,21,25]]; F,p=stats.f_oneway(*groups)
print('means',[g.mean().round(2) for g in groups],'F',round(F,3),'p',round(p,5))
y=np.concatenate(groups); grand=y.mean(); SST=np.sum((y-grand)**2); ST=sum(len(g)*(g.mean()-grand)**2 for g in groups); SE=SST-ST
print('SST',round(SST,1),'treatment SS',round(ST,1),'error SS',round(SE,1))
plt.boxplot(groups,tick_labels=['Food A','Food B','Food C']); plt.show()

### Practice set · CRD and ANOVA
1. Identify factor, treatments, experimental unit, and response in a randomized fertilizer trial.
2. With k=4 treatments and N=24 observations, give treatment, error, and total degrees of freedom.
3. If SS_T=90 and df_T=3, while SS_E=120 and df_E=20, compute F.
4. State what a significant omnibus F does and does not show.

<details><summary>Answer sketches</summary>
1. Fertilizer type; fertilizer levels; each independently randomized plot/plant as defined; growth outcome. 2. 3, 20, 23. 3. MS_T=30, MS_E=6, F=5. 4. At least one treatment mean differs; it does not identify which treatment or prove practical importance.
</details>

### Debrief
Name the experimental unit/response. What does randomization protect against?

Record the formula, assumptions, and one follow-up question.